# 08 · Regla del artefacto de comillas y envíos finales

**Objetivo:**
- implementar y evaluar la regla D34 (artefacto de comillas, H3);
- preparar los envíos candidatos;
- fijar, **antes** de ver ningún puntaje, cómo se eligen los dos envíos finales de Kaggle;
- cerrar P14 (reproducibilidad del 05).

**Punto de partida (07):**
- FINAL_A = `ens07__D__stack-all`: stacking de los 10 modelos del pool. F1 macro 0.8008 en `folds.csv` y 0.7961 en `folds_seed2`.
- FINAL_B = `V1__logreg__nested`.

**Reglas:** como en el 07, todo lo que usa etiquetas se ajusta dentro de la CV externa, y el test solo se usa para generar los envíos y comprobar tasas.

**Ejecución:** lo único costoso es reentrenar las 2 CNN para P14 (≈9 min la primera vez; quedan en la caché de `models/08_finales/cache/`). `RECOMPUTE = True` borra esa caché y las reentrena.

## 0. Configuración

In [1]:
import json
import os
import platform
import shutil
import time
import warnings

import numpy as np
import pandas as pd
import sklearn
import torch
from IPython.display import display

import ensemble as en
import modeling as md
import neural as nn_
import preprocessing as pp
from common import LABELS, PROCESSED_DIR, ROOT_DIR, SEED, TEST_PATH, TRAIN_PATH, load_test, load_train, make_folds

METRIC = "macro"  # P1 resuelta (H50)
RECOMPUTE = False  # True: borra la caché del 08 (reentrenamiento de las 2 CNN para P14, ≈9 min) y la regenera
RECOMPUTE = RECOMPUTE or os.environ.get("TWEETTOXIC_RECOMPUTE") == "1"

warnings.filterwarnings("ignore", category=UserWarning, module="torch")
nn_.set_seed(SEED)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 40)
N_JOBS, FOLD_THREADS = 5, 2
OUT_DIR = md.model_folder("08")
CACHE_DIR = OUT_DIR / "cache"
if RECOMPUTE and CACHE_DIR.exists():
    shutil.rmtree(CACHE_DIR)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_07 = md.MODELS_DIR / "07_ensamble" / "cache"
P_COLS = [f"p_{lab}" for lab in LABELS]
AG = LABELS.index("AG")

train, test = load_train(), load_test()
Y = train[LABELS].to_numpy()
texts = train["text"]
folds = make_folds(train).to_numpy()
folds2 = make_folds(train, seed=SEED + 1, path=PROCESSED_DIR / "folds_seed2.csv").to_numpy()
FOLDSETS = {"folds": folds, "folds_seed2": folds2}
EMPTY_TR = md.empty_mask(texts)

# El pool y los dos ensambles del 07 (fijados allí; no se cambian)
CORE = ["V2__logreg__nested", "V1__logreg__nested", "V3__linsvm__nested"]
CLASSIC = ["V3__et__word__k2000", "V3__xgb__word__k500", "V3__nbsvm", "V3__mnb__word"]
NETS = ["V1__nn_cnn_char__tune-d0.3-lr0.002", "V1__nn_cnn_word__w2v", "V3__nn_mlp__tune-d0.5-lr0.002"]
SUBSETS = {"C": CORE + NETS, "D": CORE + CLASSIC + NETS}
oof_store, test_store = pd.read_parquet(md.OOF_PATH), pd.read_parquet(md.TEST_PRED_PATH)


def stored(store, mid, ids):
    return store[store["model_id"] == mid].set_index("id").loc[np.asarray(ids), P_COLS].to_numpy()


P = {"folds": {m: stored(oof_store, m, train["id"]) for m in SUBSETS["D"]},
     "folds_seed2": {m: np.load(CACHE_07 / f"{m}__folds_seed2.npy") for m in SUBSETS["D"]}}
P_TEST = {m: stored(test_store, m, test["id"]) for m in SUBSETS["D"]}
print(f"Python {platform.python_version()} | scikit-learn {sklearn.__version__} | torch {torch.__version__} | "
      f"SEED = {SEED} | RECOMPUTE = {RECOMPUTE}")
print(f"Train {len(train)} | test {len(test)} | OOF de folds_seed2 leídas de la caché del 07: {len(P['folds_seed2'])}")

Python 3.14.6 | scikit-learn 1.9.1 | torch 2.14.1 | SEED = 42 | RECOMPUTE = False
Train 1731 | test 1154 | OOF de folds_seed2 leídas de la caché del 07: 10


## 1. La regla D34: definición y reproducción de H3

**Definición:** AG=1 si el **texto crudo**, tal como lo devuelven `load_train` / `load_test`, contiene `;` o `"`.

Es la misma definición del 01 §2. Un escritor de CSV pone un campo entre comillas exactamente cuando contiene el separador (`;`) o comillas, así que la regla marca los tuits que el archivo guarda entre comillas. Abajo se comprueba de las dos formas: mirando el campo en las líneas del archivo original y con la regla sobre el texto.

**Qué es y qué no es:**
- es un **artefacto de cómo se construyó el archivo**, no una propiedad del lenguaje;
- no generaliza fuera de este dataset: en otro corpus, un tuit con `;` no tiene por qué ser agresivo;
- usarla es **una decisión del equipo** (P3, D34), y así se declarará en el informe.

In [2]:
def quoted_in_file(path, n_rows):
    """¿El campo de texto de cada fila va entre comillas en el CSV original? (como en el 01 §2)."""
    lines = path.read_bytes().decode("latin-1").splitlines()[1:]
    if len(lines) != n_rows:  # alguna fila ocupa más de una línea: no se puede alinear así
        return None
    return pd.Series([line.split(";", 1)[1].startswith('"') for line in lines])


rows = []
for name, df, path in [("train", train, TRAIN_PATH), ("test", test, TEST_PATH)]:
    rule = en.quote_mask(df["text"])
    in_file = quoted_in_file(path, len(df))
    rows.append({"conjunto": name, "tuits": len(df), "con la regla (; o \")": int(rule.sum()),
                 "entre comillas en el archivo": None if in_file is None else int(in_file.sum()),
                 "coinciden": None if in_file is None else bool((in_file.to_numpy() == rule).all())})
display(pd.DataFrame(rows).set_index("conjunto"))
Q_TR, Q_TE = en.quote_mask(texts), en.quote_mask(test["text"])
assert Q_TR.sum() == 51 and Q_TE.sum() == 34, "no se reproducen los conteos de H3"
print(f"Train: de las {Q_TR.sum()} filas con la regla, {int(Y[Q_TR, AG].sum())} tienen AG=1 ({100 * Y[Q_TR, AG].mean():.0f}%) | "
      f"cubren el {100 * Q_TR[Y[:, AG] == 1].mean():.1f}% de las {int(Y[:, AG].sum())} filas con AG=1 | "
      f"AG=1 en el resto: {100 * Y[~Q_TR, AG].mean():.1f}%")
print("Combinaciones de etiquetas en las filas con la regla:",
      pd.Series(["".join(map(str, r)) for r in Y[Q_TR]]).value_counts().to_dict())

,tuits,"con la regla (; o "")",entre comillas en el archivo,coinciden
conjunto,,,,
train,1731,51,51,True
test,1154,34,34,True


Train: de las 51 filas con la regla, 51 tienen AG=1 (100%) | cubren el 7.7% de las 662 filas con AG=1 | AG=1 en el resto: 36.4%
Combinaciones de etiquetas en las filas con la regla: {'001': 26, '101': 21, '111': 4}


**Interpretación:**
- **La regla reproduce H3 exactamente:** 51 tuits en el train y 34 en el test. En los dos archivos coinciden uno a uno con los campos que el CSV original guarda entre comillas.
- **En el train es una regla de precisión total y cobertura baja:**
  - las 51 filas tienen AG=1 (100%), frente al 36.4% del resto;
  - pero solo cubren el 7.7% de las 662 filas con AG=1.
- **Combinaciones de esas filas:** 001 (26), 101 (21) y 111 (4). Las 26 con AG=1 y HS=0 son la única aparición de esa combinación en el train (H3), lo que confirma que es un patrón del archivo y no de la anotación del lenguaje.

### Implementación

`ensemble.EnsembleClassifier(..., quote_rule=True)`:
1. hace todo lo que hacía: umbrales, jerarquía HS=0 ⇒ TR=0 y tuits vacíos ⇒ 000;
2. **después** fuerza AG=1 en los tuits con la regla, sin tocar HS ni TR. AG no está sujeta a la jerarquía (D7).

El parámetro queda en el objeto guardado (`.joblib`) y en su `.json` (`quote_rule`), así que la cadena "texto crudo → etiquetas" sigue siendo reproducible desde disco.

## 2. Evaluación con el protocolo del 07

Se repite el crossfit del 07 en `folds.csv` y `folds_seed2`. Para FINAL_A se comparan sus decisiones fuera de fold con y sin la regla: F1 por etiqueta, macro y micro, y la diferencia pareada por fold.

**Importante:** en la CV, la ganancia es **por construcción**. Las 51 filas tienen todas AG=1, así que forzarlo solo puede convertir falsos negativos en aciertos. Esto **no prueba** que el test comparta el patrón; solo mide cuánto vale si lo comparte.

In [3]:
RES = {}


def crossfit(fs, key):
    if (fs, key) not in RES:
        ids = SUBSETS[key]
        RES[(fs, key)] = en.crossfit(en.Stacking(own_label_only=False), np.stack([P[fs][m] for m in ids], axis=1), Y,
                                     FOLDSETS[fs], EMPTY_TR)
    return RES[(fs, key)]


def per_fold(pred, f):
    return pd.DataFrame([en.metrics(Y[f == k], pred[f == k]) for k in sorted(np.unique(f))])


rows = []
for fs, f in FOLDSETS.items():
    pred = crossfit(fs, "D")["pred"]
    quoted = pred.copy()
    quoted[Q_TR, AG] = 1
    base, rule = per_fold(pred, f), per_fold(quoted, f)
    for tag, t in [("FINAL_A", base), ("FINAL_A_quote", rule)]:
        rows.append({"folds": fs, "modelo": tag, **{f"F1 {lab}": round(t[f"F1_{lab}"].mean(), 4) for lab in LABELS},
                     "F1 macro": round(t["F1_macro"].mean(), 4), "desv.": round(t["F1_macro"].std(), 4),
                     "F1 micro": round(t["F1_micro"].mean(), 4)})
    d = en.paired_delta(rule["F1_macro"], base["F1_macro"])
    dm = en.paired_delta(rule["F1_micro"], base["F1_micro"])
    rows.append({"folds": fs, "modelo": "Δ (con − sin regla)",
                 "F1 macro": f"{d['Δ']:+.4f} ± {d['Δ desv.']:.4f} ({d['folds a favor']}/5)",
                 "F1 micro": f"{dm['Δ']:+.4f} ± {dm['Δ desv.']:.4f} ({dm['folds a favor']}/5)",
                 "F1 AG": f"{(rule['F1_AG'] - base['F1_AG']).mean():+.4f}"})
    print(f"{fs}: de las 51 filas con la regla, FINAL_A ya predecía AG=1 en {int(pred[Q_TR, AG].sum())}; la regla cambia {int((quoted != pred).sum())}")
display(pd.DataFrame(rows).set_index(["folds", "modelo"]).fillna(""))

folds: de las 51 filas con la regla, FINAL_A ya predecía AG=1 en 33; la regla cambia 18


folds_seed2: de las 51 filas con la regla, FINAL_A ya predecía AG=1 en 31; la regla cambia 20


F1 HS   F1 TR    F1 AG                F1 macro   desv.                F1 micro
folds       modelo                                                                                              
folds       FINAL_A              0.7886  0.8403   0.7735                  0.8008   0.018                  0.7968
            FINAL_A_quote        0.7886  0.8403   0.7892                   0.806   0.016                  0.8022
            Δ (con − sin regla)                  +0.0157  +0.0052 ± 0.0025 (5/5)          +0.0054 ± 0.0026 (5/5)
folds_seed2 FINAL_A              0.7913  0.8356   0.7613                  0.7961  0.0212                  0.7927
            FINAL_A_quote        0.7913  0.8356   0.7789                  0.8019  0.0223                  0.7987
            Δ (con − sin regla)                  +0.0176  +0.0059 ± 0.0022 (5/5)          +0.0060 ± 0.0024 (5/5)

**Interpretación:**
- **En `folds.csv`:**
  - FINAL_A ya predecía AG=1 en 33 de las 51 filas; la regla cambia las 18 restantes;
  - el F1 de AG pasa de 0.7735 a 0.7892 (+0.0157);
  - el F1 macro sube +0.0052 ± 0.0025 (5/5 folds), de 0.8008 a 0.8060; la micro, +0.0054.
- **En `folds_seed2`:** cambian 20 filas, AG +0.0176 y F1 macro +0.0059 ± 0.0022 (5/5), de 0.7961 a 0.8019.
- **HS y TR no cambian.**
- **Recordatorio:** es una ganancia **por construcción**. En el train las 51 filas son todas AG=1, así que la regla solo puede acertar. Mide cuánto valdría si el test tiene el mismo patrón; **no prueba** que lo tenga.

## 3. Envíos candidatos

| Envío | Modelo | Regla |
|---|---|---|
| FINAL_A (ya existe, del 07) | `ens07__D__stack-all`: stacking de los 10 modelos | no |
| **FINAL_A_quote** | el mismo modelo, cargado del 07, con `quote_rule=True` | sí |
| **FINAL_C** | el otro ensamble que cumplió la regla del 07 en ambos conjuntos de folds: C = núcleo lineal + redes, stacking de 3 etiquetas | no |

FINAL_C se ajusta como FINAL_A en el 07: el combinador con todas las OOF de `folds.csv` y los umbrales de meseta sobre sus predicciones fuera de fold.

`make_submission` comprueba filas, ids y su orden, enteros 0/1 y nulos.

In [4]:
SUB_DIR = md.SUBMISSIONS_DIR
FINAL_A = md.load_model("ens07__D__stack-all")
assert not getattr(FINAL_A, "quote_rule", False)

# FINAL_A_quote: el mismo combinador y umbrales del 07 + la regla
QUOTE_NAME = "ens08__D__stack-all__quote"
final_quote = en.EnsembleClassifier(FINAL_A.base_ids, FINAL_A.combiner, FINAL_A.thresholds, name=QUOTE_NAME,
                                    quote_rule=True)
cv_d = {fs: {k: round(v, 4) for k, v in per_fold(crossfit(fs, "D")["pred"], f).mean().items()} for fs, f in FOLDSETS.items()}
en.save_ensemble(final_quote, QUOTE_NAME, {"notebook": "08", "role": "FINAL_A_quote (D34)", "subset": "D", "method": "stack-all",
                                           "same_as": "ens07__D__stack-all + quote_rule", "cv_without_rule": cv_d, "seed": SEED})

# FINAL_C: subconjunto C con el mismo método
C_NAME = "ens08__C__stack-all"
P_C = np.stack([P["folds"][m] for m in SUBSETS["C"]], axis=1)
res_c = crossfit("folds", "C")
fitted_c, thr_c = en.fit_final(en.Stacking(own_label_only=False), P_C, Y, folds, res_c["proba"])
final_c = en.EnsembleClassifier(SUBSETS["C"], fitted_c, thr_c, name=C_NAME)
cv_c = {fs: {k: round(v, 4) for k, v in per_fold(crossfit(fs, "C")["pred"], f).mean().items()} for fs, f in FOLDSETS.items()}
path_c = en.save_ensemble(final_c, C_NAME, {"notebook": "08", "role": "FINAL_C", "subset": "C", "method": "stack-all",
                                            "cv": cv_c, "seed": SEED})
print(f"FINAL_C: umbrales {thr_c} | F1 macro {cv_c['folds']['F1_macro']:.4f} (folds) | {cv_c['folds_seed2']['F1_macro']:.4f} (folds_seed2)")

files = {"FINAL_A": SUB_DIR / "FINAL_A_ens07__D__stack-all.csv",
         "FINAL_A_quote": md.make_submission(QUOTE_NAME, filename=f"FINAL_A_quote_{QUOTE_NAME}"),
         "FINAL_C": md.make_submission(C_NAME, filename=f"FINAL_C_{C_NAME}")}
SUBS = {k: pd.read_csv(v) for k, v in files.items()}

# FINAL_A recalculado ahora coincide con el archivo del 07
pred_a = md.decide(FINAL_A.predict_proba_from_base(np.stack([P_TEST[m] for m in FINAL_A.base_ids], axis=1)),
                   test["text"], FINAL_A.thresholds, FINAL_A.hierarchy, FINAL_A.empty_as_negative)
assert (SUBS["FINAL_A"][LABELS].to_numpy() == pred_a).all()

# Los archivos con y sin regla solo difieren en AG de las filas con la regla
diff = SUBS["FINAL_A_quote"][LABELS].to_numpy() != SUBS["FINAL_A"][LABELS].to_numpy()
assert not diff[:, :AG].any() and not diff[~Q_TE].any(), "difieren fuera de AG o fuera de las filas con la regla"
assert (SUBS["FINAL_A_quote"].loc[Q_TE, "AG"] == 1).all()
CHANGED = diff[:, AG]
rows = [{"envío": k, "archivo": f"{v.parent.name}/{v.name}", "filas": len(SUBS[k]),
         "ids en orden": bool((SUBS[k]["id"] == test["id"]).all()), "valores": sorted(np.unique(SUBS[k][LABELS]).tolist()),
         "nulos": int(SUBS[k].isna().sum().sum()), **{f"% {lab}=1": round(100 * SUBS[k][lab].mean(), 1) for lab in LABELS}}
        for k, v in files.items()]
display(pd.DataFrame(rows).set_index("envío"))
print(f"Test: {Q_TE.sum()} tuits con la regla | FINAL_A ya predecía AG=1 en {int(pred_a[Q_TE, AG].sum())} | "
      f"FINAL_A_quote cambia {int(CHANGED.sum())} (solo AG) | ids que cambian: {test.loc[CHANGED, 'id'].tolist()}")
d_ac = (SUBS["FINAL_C"][LABELS].to_numpy() != SUBS["FINAL_A"][LABELS].to_numpy())
print(f"FINAL_C frente a FINAL_A: difieren {int(d_ac.any(axis=1).sum())} tuits "
      f"({', '.join(f'{lab}: {int(d_ac[:, j].sum())}' for j, lab in enumerate(LABELS))})")

FINAL_C: umbrales {'HS': 0.39, 'TR': 0.36, 'AG': 0.37} | F1 macro 0.7965 (folds) | 0.7981 (folds_seed2)


,archivo,filas,ids en orden,valores,nulos,% HS=1,% TR=1,% AG=1
envío,,,,,,,,
FINAL_A,submissions/FINAL_A_ens07__D__stack-all.csv,1154,True,"[0, 1]",0,46.6,30.9,42.3
FINAL_A_quote,submissions/FINAL_A_quote_ens08__D__stack-all__quote.csv,1154,True,"[0, 1]",0,46.6,30.9,42.7
FINAL_C,submissions/FINAL_C_ens08__C__stack-all.csv,1154,True,"[0, 1]",0,46.7,29.9,42.7


Test: 34 tuits con la regla | FINAL_A ya predecía AG=1 en 29 | FINAL_A_quote cambia 5 (solo AG) | ids que cambian: [21757, 23138, 20849, 24690, 24805]
FINAL_C frente a FINAL_A: difieren 91 tuits (HS: 41, TR: 14, AG: 51)


In [5]:
# Registro y predicciones: FINAL_C es un ensamble nuevo. FINAL_A_quote tiene las mismas probabilidades que FINAL_A
# (solo cambia la decisión en AG), así que no se registra aparte: queda en su .json.
t = res_c["folds"][("plateau", True)]
proba_test_c = final_c.predict_proba_from_base(np.stack([P_TEST[m] for m in SUBSETS["C"]], axis=1))
row = {"model_id": C_NAME, "family": "ensamble", "classifier": "stack-all", "variant": "—",
       "features": "subconjunto C: " + " + ".join(SUBSETS["C"]),
       "params": json.dumps({k: v for k, v in final_c.combiner.describe().items() if k != "coef"}), "notebook": "08",
       "F1_macro": round(t["F1_macro"].mean(), 4), "F1_macro_std": round(t["F1_macro"].std(), 4),
       "F1_micro": round(t["F1_micro"].mean(), 4), **{f"F1_{lab}": round(t[f"F1_{lab}"].mean(), 4) for lab in LABELS},
       **{f"thr_{lab}": thr_c[lab] for lab in LABELS}, "proba_calibrated": True, "cv_seconds": None,
       "size_mb": round(path_c.stat().st_size / 1e6, 3), "saved": True, "architecture": f"{len(SUBSETS['C'])} modelos base",
       "seeds": None, "epochs": None, "train_seconds": None}
reg = md.update_registry([row])
md.update_predictions([md.prediction_frame(C_NAME, train["id"], res_c["proba"], folds, calibrated=True)], md.OOF_PATH)
md.update_predictions([md.prediction_frame(C_NAME, test["id"], proba_test_c, pred=SUBS["FINAL_C"][LABELS].to_numpy(),
                                           calibrated=True)], md.TEST_PRED_PATH)
print(f"Registro: {len(reg)} modelos | añadido: {C_NAME}")

Registro: 187 modelos | añadido: ens08__C__stack-all


**Interpretación:**
- **FINAL_A recalculado ahora** a partir de las probabilidades guardadas coincide con el archivo del 07.
- **FINAL_A_quote frente a FINAL_A:**
  - de los 34 tuits del test con la regla, FINAL_A ya predecía AG=1 en 29;
  - la regla cambia **solo 5 tuits, y solo en AG** (de 0 a 1), comprobado con `assert`;
  - el % de AG=1 en el test pasa de 42.3 a 42.7.
- **FINAL_C** (núcleo lineal + redes, stacking de 3 etiquetas):
  - umbrales 0.39 / 0.36 / 0.37;
  - F1 macro 0.7965 en `folds.csv` y 0.7981 en `folds_seed2`;
  - se diferencia de FINAL_A en 91 tuits (HS 41, TR 14, AG 51), así que es una alternativa realmente distinta;
  - queda registrado (187 modelos) con sus predicciones.
- **Comprobaciones:** los tres archivos pasan todas (1154 filas, ids en orden, valores 0/1, sin nulos).

## 4. Marco de decisión de los dos envíos finales (fijado antes de subir nada)

Kaggle puntúa la competencia con **el mejor de los dos envíos seleccionados**, así que los dos finales deben cubrir la incertidumbre que queda. La principal es si el test comparte el patrón de comillas.

**Árbol de decisión** (Δ = LB público de FINAL_A_quote − LB público de FINAL_A):

| Caso | Condición | Interpretación | Envíos finales |
|---|---|---|---|
| a) | Δ ≥ +0.003 | la regla se valida en la parte pública | FINAL_A_quote + FINAL_A (seguro por si la parte privada difiere) |
| b) | −0.003 < Δ < +0.003 | no concluyente | FINAL_A_quote + FINAL_A |
| c) | Δ ≤ −0.003 | la regla no se sostiene | FINAL_A + FINAL_C |

**Por qué esta diferencia sí es limpia:** los dos archivos son idénticos salvo en AG de los tuits con la regla que FINAL_A no predecía ya como AG=1 (comprobado con `assert` en la sección 3). Cualquier diferencia de puntaje se debe solo a esas filas.

**Por qué no se usa el leaderboard para elegir entre modelos distintos (H44):**
- la parte pública es una fracción del test (≈1150 tuits en total);
- dos modelos distintos difieren en cientos de decisiones, así que una diferencia de ≈0.01 está dentro del ruido de esa muestra;
- elegir el modelo por el leaderboard sería ajustar a ese ruido. Por eso FINAL_A, FINAL_B y FINAL_C se eligieron por CV.

**Cuánto puede moverse Δ como máximo:** con pocas filas cambiadas, la diferencia está acotada. La tabla de abajo estima la mayor ganancia y la mayor pérdida posibles si todas las filas cambiadas caen en la parte puntuada:
- usa las tasas de aciertos y errores de AG de FINAL_A en las OOF, escaladas al tamaño de la parte puntuada;
- se calcula para el test completo (1154 tuits) y para la mitad, porque el tamaño de la parte pública no se conoce.

In [6]:
def lb_bounds(n_scored, k_changed):
    """Rango de Δ F1 macro si las k filas cambiadas (AG 0 -> 1) están en la parte puntuada: todas aciertos
    (FN -> VP) o todas errores (VN -> FP). Conteos de AG escalados desde las OOF de FINAL_A."""
    pred, y = crossfit("folds", "D")["pred"][:, AG], Y[:, AG]
    s = n_scored / len(y)
    tp, fp, fn = s * ((pred == 1) & (y == 1)).sum(), s * ((pred == 1) & (y == 0)).sum(), s * ((pred == 0) & (y == 1)).sum()
    f1 = lambda tp_, fp_, fn_: 2 * tp_ / (2 * tp_ + fp_ + fn_)
    base = f1(tp, fp, fn)
    return {"tuits puntuados": n_scored, "filas cambiadas": k_changed,
            "Δ macro si todas aciertan": round((f1(tp + k_changed, fp, fn - k_changed) - base) / 3, 4),
            "Δ macro si todas fallan": round((f1(tp, fp + k_changed, fn) - base) / 3, 4)}


k = int(CHANGED.sum())
bounds = pd.DataFrame([lb_bounds(n, k) for n in (len(test), len(test) // 2)]).set_index("tuits puntuados")
display(bounds)


def finals_from_lb(lb_quote, lb_a, margin=0.003):
    """Aplica el árbol de decisión con los puntajes públicos de FINAL_A_quote y FINAL_A."""
    d = lb_quote - lb_a
    if d >= margin:
        return "a) la regla se valida en la parte pública", ["FINAL_A_quote", "FINAL_A"]
    if d <= -margin:
        return "c) la regla no se sostiene", ["FINAL_A", "FINAL_C"]
    return "b) no concluyente", ["FINAL_A_quote", "FINAL_A"]


LB = {"FINAL_A": 0.78273, "FINAL_A_quote": 0.78598}  # puntajes públicos del equipo (PROGRESO §8), anotados al subirlos
if LB["FINAL_A_quote"] is None:
    print("Pendiente: subir FINAL_A_quote y anotar su puntaje público; después se aplica finals_from_lb().")
else:
    case, finals = finals_from_lb(LB["FINAL_A_quote"], LB["FINAL_A"])
    print(f"Δ público = {LB['FINAL_A_quote']:.5f} − {LB['FINAL_A']:.5f} = {LB['FINAL_A_quote'] - LB['FINAL_A']:+.5f} -> {case} | "
          f"envíos finales: {' + '.join(finals)}")

,filas cambiadas,Δ macro si todas aciertan,Δ macro si todas fallan
tuits puntuados,,,
1154,5,0.0022,-0.0014
577,5,0.0044,-0.0027


Δ público = 0.78598 − 0.78273 = +0.00325 -> a) la regla se valida en la parte pública | envíos finales: FINAL_A_quote + FINAL_A


**Qué implica la cota para el árbol (observación, no cambia el árbol fijado):** con solo 5 filas cambiadas, Δ está acotado.
- **Si se puntuara el test completo:** entre −0.0014 (las 5 mal) y +0.0022 (las 5 bien).
- **Si la parte pública es la mitad:** entre −0.0027 y +0.0044, y solo si las 5 caen en ella.
- **Δ = 0** si ninguna de las 5 cae en la parte pública.

Consecuencias:
- **c) (Δ ≤ −0.003) no se puede alcanzar** ni en el peor caso, así que con el margen de ±0.003 el árbol no elegirá nunca FINAL_C;
- **a)** solo es posible si casi todas las filas cambiadas están en la parte pública y son aciertos;
- **lo más probable es b):** finales FINAL_A_quote + FINAL_A.

Si el equipo quiere que FINAL_C sea alcanzable, la condición de c) tendría que ser menos exigente (por ejemplo, Δ < 0). Esa decisión es del equipo y aquí no se cambia.

**Resultado (puntajes públicos del 2026-10-15):** FINAL_A_quote 0.78598 frente a FINAL_A 0.78273, Δ = **+0.00325**.
- **Es el caso a):** la regla se valida en la parte pública. **Envíos finales: FINAL_A_quote + FINAL_A.**
- **Δ está cerca de la cota máxima** (+0.0044 si se puntúa la mitad del test y las 5 filas aciertan). Lo más compatible es que la mayoría de los 5 tuits cambiados estén en la parte pública y sean de verdad AG=1, es decir, que la parte pública del test comparte el patrón de comillas.
- **Aun así, son a lo sumo 5 tuits.** FINAL_A se mantiene como segundo envío por si la parte privada no lo comparte.
- **FINAL_C** sacó 0.77760 (−0.005 frente a FINAL_A). Es una diferencia entre modelos distintos y está dentro del ruido del leaderboard (H44), así que no se usa para nada.

## 5. Cierre de P14: reproducción de las 2 CNN del ensamble

El 05 solo se ejecutó una vez, y repetirlo entero cuesta ≈3 h. En su lugar, se reentrenan con su configuración guardada y las mismas semillas (5 folds × 3 semillas, early stopping interno) las dos CNN que forman parte del ensamble final, y se comparan sus OOF con las almacenadas (tolerancia 1e-6).

In [7]:
rows = []
for mid in ["V1__nn_cnn_char__tune-d0.3-lr0.002", "V1__nn_cnn_word__w2v"]:
    params = md.load_model(mid).get_params()
    params.update(fixed_epochs=None, thresholds=0.5, n_threads=FOLD_THREADS)
    print(f"{mid} ...", end=" ", flush=True)
    start = time.perf_counter()
    cv = nn_.cached_cross_validate(nn_.NeuralTweetClassifier(**params), texts, Y, pd.Series(folds), CACHE_DIR / f"{mid}__folds",
                                   n_jobs=N_JOBS)
    secs = time.perf_counter() - start
    diff = float(np.abs(cv["oof"] - P["folds"][mid]).max())
    src = "caché" if cv["cached_folds"] == 5 else "reentrenada"
    print(f"máx. |diferencia| {diff:.2e} | {src} | {secs:.0f} s (entrenamiento: {cv['fit_seconds']:.0f} s)", flush=True)
    rows.append({"modelo": mid, "semillas": params["n_seeds"], "máx. |diferencia| con la OOF del 05": diff,
                 "reproduce (≤ 1e-6)": diff <= 1e-6, "tiempo de entrenamiento (s)": round(cv["fit_seconds"])})
p14 = pd.DataFrame(rows).set_index("modelo")
display(p14)
assert p14["reproduce (≤ 1e-6)"].all()

V1__nn_cnn_char__tune-d0.3-lr0.002 ... 

máx. |diferencia| 0.00e+00 | caché | 4 s (entrenamiento: 331 s)


V1__nn_cnn_word__w2v ... 

máx. |diferencia| 0.00e+00 | caché | 4 s (entrenamiento: 144 s)


,semillas,máx. |diferencia| con la OOF del 05,reproduce (≤ 1e-6),tiempo de entrenamiento (s)
modelo,,,,
V1__nn_cnn_char__tune-d0.3-lr0.002,3,0.0,True,331
V1__nn_cnn_word__w2v,3,0.0,True,144


**Interpretación:**
- **Las dos CNN se reproducen exactamente:** diferencia máxima de 0.0 con las OOF del 05 (tolerancia 1e-6), con 3 semillas × 5 folds.
- **Coste:** ≈5 min de entrenamiento para la de caracteres y ≈2 min para la de palabras.
- **P14 queda resuelta, con un matiz:** el 05 completo no se volvió a ejecutar (≈3 h), pero sí se reprodujeron todos los modelos del 05 que usa el ensamble final:
  - las dos CNN, aquí;
  - el MLP, en el 07 §5a.
- **El resto de modelos del pool** son de scikit-learn, xgboost y LightGBM. Cuatro de ellos ya se reprodujeron en el 07, y las logísticas anidadas se recalcularon en `folds_seed2` con la misma rejilla.

## 6. Conclusiones

1. **La regla D34 reproduce H3:** 51 tuits en el train (todos con AG=1; cubren el 7.7% de los AG=1) y 34 en el test, idénticos a los campos entre comillas del CSV.
2. **En la CV**, aplicada a FINAL_A:
   - sube el F1 macro +0.0052 (5/5) en `folds.csv` y +0.0059 (5/5) en `folds_seed2`;
   - la ganancia es por construcción y no prueba que el test comparta el patrón.
3. **En el test solo cambia 5 tuits:** FINAL_A ya predecía AG=1 en 29 de los 34. FINAL_A_quote y FINAL_A difieren únicamente en AG de esas 5 filas.
4. **Envíos generados:** `FINAL_A_quote_ens08__D__stack-all__quote.csv` y `FINAL_C_ens08__C__stack-all.csv` (0.7965 / 0.7981 en CV). FINAL_A ya existía.
5. **Árbol de decisión:**
   - fijado antes de subir los envíos: a) y b) → FINAL_A_quote + FINAL_A; c) → FINAL_A + FINAL_C;
   - con 5 filas, Δ está acotado entre ≈−0.003 y ≈+0.004, así que c) es inalcanzable;
   - con los puntajes públicos, Δ = +0.00325: caso a). **Envíos finales: FINAL_A_quote + FINAL_A.**
6. **P14 resuelta:** las 2 CNN del ensamble se reproducen exactamente. El 05 completo no se volvió a ejecutar.